# NB00 — Setup & Validator Smoke Test

**Stage S0** of `Plan_ML_Challenge.md`. Exit criterion: a dummy all-empty
submission passes `utils/validate_submission.py`.

This notebook is the environment sanity check every later notebook depends on. It:

1. Detects whether it is running locally or on SageMaker and sets paths accordingly.
2. Verifies `src/business_entity_resolution` imports cleanly.
3. Peeks at each dataset file (row counts, first row).
4. Writes an all-singletons `matching_results.tsv` + `candidate_pairs.tsv`.
5. Runs the provided validator and expects `PASS`.

Copy-paste into a SageMaker notebook instance unchanged — the config cell
auto-detects the environment (D-14). See `DECISIONS.md` D-17 for why logic lives
in `src/` rather than in this notebook.


## 1. Config cell (identical across all notebooks)


In [ ]:
# --- config cell: identical across notebooks ---
import os, sys, pathlib
ON_SAGEMAKER = pathlib.Path("/opt/ml").exists() or "SM_CURRENT_HOST" in os.environ
ROOT = pathlib.Path(os.environ.get(
    "BER_ROOT",
    "/home/ec2-user/SageMaker/ber" if ON_SAGEMAKER
    else "/Users/pocketfm/Desktop/amazon_ML/Working Folder"
))
DATA, ART, OUT = ROOT/"student_resource/dataset", ROOT/"artifacts", ROOT/"output"
sys.path.insert(0, str(ROOT/"src"))
for p in (ART, OUT): p.mkdir(parents=True, exist_ok=True)
N_JOBS = os.cpu_count() or 1
print(f"ON_SAGEMAKER={ON_SAGEMAKER}  ROOT={ROOT}  N_JOBS={N_JOBS}")


## 2. Import from `src/`


In [ ]:
from business_entity_resolution.config import detect_paths, is_sagemaker, n_jobs
from business_entity_resolution.io_tsv import read_source_ids, write_submission_tsv
from business_entity_resolution.submission import (
    write_pair_of_tsvs, write_empty_submission, run_validator,
)

paths = detect_paths()
print("paths.root      :", paths.root)
print("paths.data      :", paths.data)
print("paths.output    :", paths.output)
print("paths.validator :", paths.validator)
assert paths.data.exists(),      f"dataset dir missing: {paths.data}"
assert paths.validator.exists(), f"validator missing: {paths.validator}"


## 3. Peek at each dataset file

Counts only — no full loads at S0.


In [ ]:
import subprocess

def peek(path):
    with open(path, encoding="utf-8") as f:
        header = next(f).rstrip("\n")
        first  = next(f, "").rstrip("\n")
    n = int(subprocess.check_output(["wc", "-l", str(path)]).split()[0]) - 1
    print(f"  {path.name:28s}  rows={n:>10,}  cols={header.split(chr(9))}")
    print(f"      first: {first[:120]!r}")

print("TRAIN:")
for name in ("train_source1.tsv","train_source2.tsv","train_source3.tsv","train_ground_truth.tsv"):
    peek(paths.train / name)
print("TEST:")
for name in ("test_source1.tsv","test_source2.tsv","test_source3.tsv"):
    peek(paths.test / name)


## 4. Emit the all-empty submission

Every test S1 entity gets an empty match list. This is not a real submission — it is
the format smoke test (D-18). It also gives us the local baseline of ~0.056 (`§A.2`)
the moment we upload it in S2.


In [ ]:
n_m, n_c = write_empty_submission(paths)
print(f"wrote {n_m:,} matching rows and {n_c:,} candidate rows")

# Peek the head/tail so we can eyeball the exact bytes going to disk.
for name in ("matching_results.tsv", "candidate_pairs.tsv"):
    p = paths.output / name
    print(f"\n--- {name}  ({p.stat().st_size:,} bytes) ---")
    with open(p, encoding="utf-8") as f:
        head = [next(f) for _ in range(3)]
    for line in head:
        print(repr(line))


## 5. Run the validator — the S0 exit criterion


In [ ]:
rc, out = run_validator(paths, check_ids=False)
print(out)
assert rc == 0, f"validator failed with exit {rc}"
print("S0 exit criterion met: PASS")


## 6. Next

S0 complete. Next stage is **S1 — exact macro-F_0.5 + stratified holdout by
country × cardinality** (`NB02_metric_and_splits.ipynb`). Exit criterion: unit test
reproduces the statement's worked example (0.714).

Do **not** upload this empty submission — that is S2's job, run after S1 confirms the
metric implementation. Uploading an untested baseline burns a submission slot.
